### 1. Imports

In [1]:
import os
import requests
from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

### 2. Configurations

In [2]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-11\key-vault\openai\api.key") as f:
    openai_api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

In [3]:
CIS_URL = r"http://localhost:7860/api/v1/run/eef87fb7-0ffe-4987-9be8-2e17813b7eb0"

In [4]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-9\key-vault\nvd-database\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

In [5]:
MODEL = "gpt-4.1-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)

### 3. Build the Tools

#### CIS Tool

In [6]:
import os

import requests

url = "http://localhost:7860/api/v2/workflows"
payload = {
    "flow_id": "eef87fb7-0ffe-4987-9be8-2e17813b7eb0",
    "input_value": "what are the windows password requirements for a strong password?"
}
headers = {
    "Content-Type": "application/json",
    "x-api-key": "sk-qNz-70oQVfAE8wvyRadRAVpwVDbfBtxcj_ivZNHx_Ow",
}

response = requests.post(url, json=payload, headers=headers)
response.raise_for_status()
result = response.json()

# The text reply is in output.text:
print(result["output"]["text"])

##Direct Answer
The Windows password requirements for a strong password include:
- A password must be at least six characters in length.
- It must contain characters from three of the following categories:
  - English uppercase characters (A through Z)
  - English lowercase characters (a through z)
  - Base 10 digits (0 through 9)
  - Non-alphabetic characters (such as !, $, #, %)
  - Any Unicode character that does not fall under the previous four categories.

Best practices suggest using a minimum of an 8-character password for accounts using multi-factor authentication (MFA) and a 14-character password for accounts not using MFA.

##CIS Sections
- 5.2 Use Unique Passwords
- 1.1.4 Ensure 'Minimum password length' is set to '14 or more character(s)'
- 18.9.26.5 Ensure 'Password Settings: Password Length' is set to 'Enabled: 15 or more'

##Explanation
The password complexity requirements are designed to enhance security by making passwords harder to guess or crack. The requirement for 

In [7]:
import requests
from langchain_core.tools import tool


@tool
def ask_cis(question: str) -> str:
    """Query the Flowise workflow knowledge base for CIS benchmarks and security guidance.

    Args:
        question: The security question or topic to search.
    """
    url = "http://localhost:7860/api/v2/workflows"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": "sk-qNz-70oQVfAE8wvyRadRAVpwVDbfBtxcj_ivZNHx_Ow",
    }
    payload = {
        "flow_id": "eef87fb7-0ffe-4987-9be8-2e17813b7eb0",
        "input_value": question,
    }

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        result = response.json()

        # Safely extract output text
        return result.get("output", {}).get("text", "No output returned.")
    except requests.RequestException as e:
        return f"Error executing CIS workflow request: {e}"

In [8]:
ask_cis.invoke("Tell me about proper shutdown of windows servers")

"##Direct Answer\nProper shutdown of Windows servers should involve using the appropriate user rights to initiate a shutdown command, ensuring that only authorized users can perform this action to prevent denial-of-service (DoS) conditions.\n\n##CIS Sections\n- **2.2.21 Ensure 'Force shutdown from a remote system' is set to 'Administrators'**\n- **2.2.37 Ensure 'Shut down the system' is set to 'Administrators, Users'**\n\n##Explanation\nThe policy setting for 'Force shutdown from a remote system' recommends that only trusted administrators have the ability to shut down a Windows server remotely. This restriction helps prevent unauthorized shutdowns that could lead to service interruptions or DoS conditions. \n\nOn the other hand, the 'Shut down the system' setting allows both Administrators and authorized Users to shut down the server. This setting should be carefully managed to prevent guests or unauthorized users from executing shutdown commands, which could also result in DoS condit

#### NVD Tool

In [9]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [10]:
lookup_cve.invoke("SMB")

[TOOL]CVE Tool Activated


'CVE-1999-1387: Windows NT 4.0 SP2 allows remote attackers to cause a denial of service (crash), possibly via malformed inputs or packets, such as those generated by a Linux smbmount command that \nCVE-1999-0225: Windows NT 4.0 allows remote attackers to cause a denial of service via a malformed SMB logon request in which the actual data size does not match the specified size.\nCVE-1999-0495: A remote attacker can gain access to a file system using ..  (dot dot) when accessing SMB shares.'

#### IP Config Tool

In [11]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [12]:
print(ipconfig_tool.invoke({}))


Windows IP Configuration


Ethernet adapter Ethernet:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . : 

Unknown adapter Local Area Connection:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . : 

Wireless LAN adapter Local Area Connection* 1:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . : 

Wireless LAN adapter Local Area Connection* 2:

   Media State . . . . . . . . . . . : Media disconnected
   Connection-specific DNS Suffix  . : 

Wireless LAN adapter Wi-Fi:

   Connection-specific DNS Suffix  . : lan
   IPv6 Address. . . . . . . . . . . : 2409:40f2:159:1b59:cd19:af2d:c995:ba0a
   Temporary IPv6 Address. . . . . . : 2409:40f2:159:1b59:1d41:8694:8aa:1a3
   Temporary IPv6 Address. . . . . . : 2409:40f2:159:1b59:203c:c13:2c34:cfd9
   Temporary IPv6 Address. . . . . . : 2409:40f2:159:1b59:7d1b:f3f3:6abf:b01f
   Temporary IPv6 Address. . . 

### 4. Agent Layer

In [13]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [14]:
retrieval_agent = create_agent(
    model=llm,
    tools=[ask_cis],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [15]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [16]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

In [17]:
response = planner.invoke({ 'messages': 'what is the password requriement for windows?'})

In [21]:
print(response['messages'][1].content)

For the request about password requirements for Windows, a CVE lookup is not required since this is about general security policy rather than a specific vulnerability. However, RAG (Retrieval-Augmented Generation) could be useful to provide the most up-to-date and detailed password policy information from official Microsoft documentation.

Decision:
- RAG: Yes
- CVE lookup: No


### 5. State

In [22]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

### 6. Nodes

In [23]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [24]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [25]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [26]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

### 7. Workflow

In [27]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [28]:
app = graph.compile()

### 8. Execute

In [29]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

In [30]:
result

{'query': 'How can I harden Windows SMB services against ransomware?',
 'plan': "{'messages': [HumanMessage(content='How can I harden Windows SMB services against ransomware?', additional_kwargs={}, response_metadata={}, id='f175e5cb-5d64-41b6-9774-36d499f7332b'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be useful here to provide the most up-to-date and specific security measures and known vulnerabilities related to Windows SMB services.\\n\\nI will proceed with RAG and CVE lookup to gather detailed and current information on hardening Windows SMB services against ransomware.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 37, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_toke

In [31]:
print(result["final"]["messages"][-1].content)

To harden Windows SMB services against ransomware, implement the following validated best practices:

1. **Disable SMBv1**: SMBv1 is outdated and vulnerable; disabling it reduces the attack surface significantly.
2. **Enforce Minimum SMB Version 3.1.1**: Configure systems to require SMBv3.1.1 or higher, which supports stronger encryption and security features.
3. **Enable SMB Signing**: This authenticates both clients and servers, preventing man-in-the-middle and session hijacking attacks.
4. **Audit Unencrypted SMB Traffic**: Enable auditing to detect and log any SMB traffic that is not encrypted, helping identify potential security risks.
5. **Apply Security Patches Promptly**: Keep Windows and SMB services fully updated with the latest security patches.
6. **Restrict SMB Access**: Limit SMB access to only necessary users and systems; use firewalls to block SMB traffic from untrusted networks.
7. **Implement Network Segmentation**: Isolate critical systems to limit ransomware lateral

In [32]:
result = app.invoke({
    "query": "How can I secure windows SMB against latest attacks?"
})

[TOOL]CVE Tool Activated


In [33]:
result

{'query': 'How can I secure windows SMB against latest attacks?',
 'plan': "{'messages': [HumanMessage(content='How can I secure windows SMB against latest attacks?', additional_kwargs={}, response_metadata={}, id='972212a2-d50e-4d1d-9c29-977c208acdfa'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be required to provide the most up-to-date and detailed information on securing Windows SMB against the latest attacks.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 36, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_bcb4333501', 'id': 'chatcmpl-EG1RIkjLjus

In [34]:
print(result["final"]["messages"][-1].content)

Final Answer:

To secure Windows SMB against the latest attacks, follow these validated best practices based on current CIS benchmarks and known vulnerabilities:

1. **Set the minimum SMB protocol version to SMB v3.1.1**  
   This is the most secure SMB version, offering encryption and improved security features.

2. **Disable SMBv1**  
   SMBv1 is outdated and highly vulnerable to attacks such as session hijacking and replay attacks. Disabling it reduces exposure to many known exploits.

3. **Enable auditing for unencrypted SMB traffic**  
   This helps detect insecure SMB connections and potential attack attempts.

4. **Keep Windows systems fully patched**  
   Regularly apply Microsoft security updates to protect against newly discovered SMB vulnerabilities.

5. **Restrict SMB access via firewalls and network segmentation**  
   Limit SMB traffic to trusted hosts and networks only.

6. **Monitor SMB traffic for unusual activity**  
   Use logging and intrusion detection to identify 

In [35]:
Tool 3:

SyntaxError: invalid syntax (1715492085.py, line 1)

In [ ]:
Tool 4

In [ ]:
Tool 5

In [ ]:
tools = ["ask_cis", "lookup_cve", "lookup_cisa_kev", "lookup_attack", "lookup_epss"]
cyber_agent = create_agent(llm, tools=tools, system_prompt="write_your_prompt")

In [ ]:
cyber_agent.invoke(m1)